# Olshevsky et al. (2020) magnetic-null benchmark

This notebook compares **nullfinder-proto** with the public null finder of
**Chiti (2020)** (optional) against the 14 tabulated nulls in Olshevsky et al.
(2020), Appendix A. Chiti's implementation is not bundled: specify
`CHITI_PATH` to run the independent comparison.

For every method and grid resolution, we report recovered nulls, position
errors, classifications, charges, spurious detections, timings, and convergence.
A detection is recovered when matched within half a grid spacing
(`MATCH_TOL_CELLS = 0.5`). At coarse resolution, different search methods may
find different roots of the trilinear interpolant.

The exact tabulated coordinates are rounded, so positional errors below their
printed precision should not be over-interpreted.


In [ ]:
import importlib
import importlib.util
import os
import platform
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numba
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import linear_sum_assignment

# Use one canonical import name for the cached Numba field generator.
ROOT = next(
    (
        d
        for d in (Path.cwd(), Path.cwd().parent)
        if (d / "benchmarks" / "olshevsky2020_field.py").is_file()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository root or benchmarks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from benchmarks.olshevsky2020_field import (  # noqa: E402
    EXACT_NULLS as TABULATED_NULLS,
)
from benchmarks.olshevsky2020_field import (  # noqa: E402
    EXACT_TYPES as TABULATED_TYPES,
)
from benchmarks.olshevsky2020_field import (  # noqa: E402
    create_grid,
    olshevsky2020_field,
)

## 1. Configuration

* `PKG_SRC`: path to the repository's `src/` folder, or `None` for an installed package.
* `CHITI_PATH`: path to Chiti's `null_finder.py`, or `None` to run only nullfinder-proto.
* `RESOLUTIONS`: node counts per direction. At 512³, the field alone needs about
  3.2 GB in `float64`.
* `MATCH_TOL_CELLS`: threshold in cell widths for assigning recovered nulls.

Field generation and Numba compilation are excluded from search timings.
Small-grid times use the minimum of repeated runs; 512³ uses a single run.
Record the machine and software versions when publishing benchmark results.


In [ ]:
PKG_SRC = None  # e.g. Path("~/codes/nullfinder-proto/src").expanduser()
CHITI_PATH = None  # e.g. Path("~/codes/null_finder.py") from https://github.com/FedericaChiti/Null_Finder
CHITI_TOLERANCE = None  # None -> Chiti's default Newton tolerance

RESOLUTIONS = [20, 80, 512]
FIELD_DTYPE = np.float64
N_REPEAT_SMALL = 3
MATCH_TOL_CELLS = 0.5

## 2. Importing the implementations

The published `nullfinder_proto` package is the primary implementation. Chiti's
reference finder is loaded only when `CHITI_PATH` is supplied. Both are benchmarked
on the same arrays and coordinate grid.


In [ ]:
def import_package(src=None):
    if src is not None:
        location = str(Path(src).expanduser().resolve())
        if location not in sys.path:
            sys.path.insert(0, location)
    importlib.invalidate_caches()
    return importlib.import_module("nullfinder_proto")


def load_chiti(path):
    path = Path(path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(path)
    module_name = "nullfinder_benchmark_chiti"
    spec = importlib.util.spec_from_file_location(module_name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot import Chiti module from {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    if str(path.parent) not in sys.path:
        sys.path.append(str(path.parent))  # possible sibling imports
    spec.loader.exec_module(module)
    return module


t0_import = time.perf_counter()
nf_new = import_package(PKG_SRC)
new_import_s = time.perf_counter() - t0_import
print(f"nullfinder-proto {nf_new.__version__} at {nf_new.__file__}")
print(f"  import time: {new_import_s:.3f} s (possibly warmed dependencies)")

nf_fede = None
if CHITI_PATH is not None:
    t0_import = time.perf_counter()
    nf_fede = load_chiti(CHITI_PATH)
    print(f"Chiti: {CHITI_PATH} (import {time.perf_counter() - t0_import:.3f} s)")
else:
    print("Chiti: skipped (set CHITI_PATH to compare)")

BENCH_ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor() or "not reported",
    "logical_cpus": os.cpu_count(),
    "numba_threads": numba.get_num_threads(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "numba": numba.__version__,
    "pandas": pd.__version__,
    "nullfinder_proto": nf_new.__version__,
    "chiti_path": str(CHITI_PATH) if CHITI_PATH is not None else None,
    "match_tol_cells": MATCH_TOL_CELLS,
    "resolutions": tuple(RESOLUTIONS),
}
print("\nBenchmark environment/configuration:")
for key, value in BENCH_ENV.items():
    print(f"  {key}: {value}")

## 3. A common interface for all implementations

Each runner takes `(bx, by, bz, x, y, z)` and returns a dictionary with:

* `pos`: null positions, `(N, 3)`;
* `type`: type names, among `"A"`, `"B"`, `"As"`, `"Bs"` and `"unknown"`;
* `charge`: topological charge (±1), or NaN if not provided;
* `raw`: the native output of the routine.

`run_nullfinder_api` wraps the `nullfinder_proto` API
(`find_magnetic_nulls(...)` returning `pos`, `null_type`, `null_charge`).

`run_chiti` wraps `find_nulls(nx, ny, nz, xv, yv, zv, Bx, By, Bz, tolerance)` of
Chiti Chiti's routine (Chiti, Pontin & Scott). That routine returns positions only,
so the notebook computes the type and the charge of each of its nulls:

* the Jacobian comes from the routine's own trilinear interpolant of the cell, through its
  functions `trilinear_coeffs` and `jacobian`;
* the classification rules (`classify_jacobian`) are the same as in the `nullfinder_proto`
  package.

Type differences against the other implementations therefore reflect where the nulls
are, not different classification rules.

Behaviour of `find_nulls` to keep in mind when reading the results:

* one Newton start per cell (the centre), with at most 11 iterations;
* a bilinear face test before Newton;
* nulls with z > `zv[-2]`, i.e. in the last layer of cells, are discarded on purpose
  (none of the nulls of this benchmark is there);
* a null found from two neighbouring cells is reported twice (it would appear as
  *spurious*).

It is pure Python/NumPy, so it needs no warm-up.

In [ ]:
TYPE_CODES = {0: "unknown", 1: "A", 2: "B", 3: "As", 4: "Bs"}
TYPE_ALIASES = {
    "a": "A",
    "ar": "A",
    "a-radial": "A",
    "radial a": "A",
    "negative": "A",
    "b": "B",
    "br": "B",
    "b-radial": "B",
    "radial b": "B",
    "positive": "B",
    "as": "As",
    "a-spiral": "As",
    "spiral a": "As",
    "bs": "Bs",
    "b-spiral": "Bs",
    "spiral b": "Bs",
}
TYPE_CHARGE = {"A": 1, "As": 1, "B": -1, "Bs": -1}


def normalize_type(t):
    if isinstance(t, (int, np.integer)):
        return TYPE_CODES.get(int(t), "unknown")
    return TYPE_ALIASES.get(str(t).strip().lower(), "unknown")


def standardize(pos, types, charge=None, raw=None):
    pos = np.asarray(pos, dtype=float).reshape(-1, 3)
    types = [normalize_type(t) for t in types]
    if len(types) != len(pos):
        raise ValueError("pos and types must have the same length")
    charge = np.full(len(pos), np.nan) if charge is None else np.asarray(charge, float)
    return {"pos": pos, "type": types, "charge": charge, "raw": raw}


def run_nullfinder_api(mod, bx, by, bz, x, y, z, **kw):
    res = mod.find_magnetic_nulls(bx, by, bz, x, y, z, verbose=False, **kw)
    return standardize(res["pos"], res["null_type"], res.get("null_charge"), raw=res)


def classify_jacobian(J, rel_tol=1e-10):
    # Same rules as nullfinder_proto: a clear 2+1 eigenvalue structure, i.e. fan
    # eigenvalues (a real pair, or a complex pair for spiral nulls) whose real parts
    # share one sign, and a real spine eigenvalue of the opposite sign.
    # Returns (type name, charge).
    J = np.asarray(J, dtype=float)
    charge = int(np.sign(np.linalg.det(J)))
    ev = np.linalg.eigvals(J)
    tol = rel_tol * np.max(np.abs(ev))
    cplx = np.abs(ev.imag) > tol
    if cplx.sum() == 2:
        spine_re, fan_re, spiral = ev.real[~cplx][0], ev.real[cplx][0], True
    elif cplx.sum() == 0:
        signs = np.where(ev.real > tol, 1, np.where(ev.real < -tol, -1, 0))
        if np.any(signs == 0) or abs(signs.sum()) != 1:
            return "unknown", charge
        fan_sign = signs.sum()
        spine_re = ev.real[signs == -fan_sign][0]
        fan_re = ev.real[signs == fan_sign][0]
        spiral = False
    else:
        return "unknown", charge
    if fan_re < -tol and spine_re > tol:
        return ("As" if spiral else "A"), charge
    if fan_re > tol and spine_re < -tol:
        return ("Bs" if spiral else "B"), charge
    return "unknown", charge


def run_chiti(mod, bx, by, bz, x, y, z, tolerance=CHITI_TOLERANCE):
    nulls = mod.find_nulls(
        len(x), len(y), len(z), x, y, z, bx, by, bz, tolerance=tolerance
    )
    pos = np.array([[n["x"], n["y"], n["z"]] for n in nulls], dtype=float).reshape(
        -1, 3
    )
    types, charges = [], []
    for n in nulls:
        coeffs = np.array(
            [
                mod.trilinear_coeffs(x, y, z, n["i"], n["j"], n["k"], b)
                for b in (bx, by, bz)
            ]
        )
        t, q = classify_jacobian(mod.jacobian(n["x"], n["y"], n["z"], coeffs))
        types.append(t)
        charges.append(q)
    return standardize(pos, types, charges, raw=nulls)


# name -> runner and whether it needs a warm-up call (Numba compilation) before timing
FINDERS = {
    "nullfinder-proto": {
        "run": lambda *a: run_nullfinder_api(nf_new, *a),
        "warmup": True,
    }
}
if nf_fede is not None:
    FINDERS["Chiti"] = {"run": lambda *a: run_chiti(nf_fede, *a), "warmup": False}
print("Implementations:", list(FINDERS))

## 4. Analytic benchmark field

The reference field is defined in Appendix A of Olshevsky et al. (2020).
Its implementation, tabulated positions and null types are imported from
`benchmarks/olshevsky2020_field.py`, the same module used by the benchmark scripts.

The reference implementation incorporates two corrections to the printed
formula: **Y₂ = 0.117** (rather than 0.0117), and **√30 / 7** for the constant
term in B_y. See the generator's module docstring for details.

The generator returns Fortran-contiguous arrays with shape `(nx, ny, nz)`.


In [ ]:
def create_bfield(N, dtype=FIELD_DTYPE):
    c = create_grid(N)
    return c, c.copy(), c.copy(), *olshevsky2020_field(c, c, c, dtype=dtype)

## 5. Exact nulls (Table A.2) and field residual at the tabulated positions

In [ ]:
EXACT_NULLS = pd.DataFrame(
    {
        "null_id": np.arange(1, len(TABULATED_NULLS) + 1),
        "exact_type": TABULATED_TYPES,
        "x_exact": TABULATED_NULLS[:, 0],
        "y_exact": TABULATED_NULLS[:, 1],
        "z_exact": TABULATED_NULLS[:, 2],
    }
)
exact_pos = TABULATED_NULLS.copy()


def eval_field_at_points(points):
    out = []
    for p in np.asarray(points, dtype=float):
        b = olshevsky2020_field(p[:1], p[1:2], p[2:3], dtype=np.float64)
        out.append([b[0][0, 0, 0], b[1][0, 0, 0], b[2][0, 0, 0]])
    return np.asarray(out)


B_exact = eval_field_at_points(exact_pos)
check = EXACT_NULLS.copy()
check[["Bx", "By", "Bz"]] = B_exact
check["|B|"] = np.linalg.norm(B_exact, axis=1)
check

## 6. Run all implementations

The first call of a Numba routine compiles it, so the Numba-based implementations
(`warmup=True`) are run once on a small grid before timing; Chiti's routine needs no
warm-up. For grids up to 128³ the time is the best of
`N_REPEAT_SMALL` runs. Field construction is not included in the times. If an
implementation fails, the error is recorded and the others continue.

In [ ]:
warm = create_bfield(12)
for name, spec in FINDERS.items():
    if not spec["warmup"]:
        continue
    t0 = time.perf_counter()
    try:
        spec["run"](*warm[3:], *warm[:3])
        print(f"{name:10s} compiled / warmed up in {time.perf_counter() - t0:6.1f} s")
    except Exception as e:
        print(f"{name:10s} FAILED on warm-up: {type(e).__name__}: {e}")

RESULTS = {}  # (name, N) -> standardized result
TIMES = {}  # (name, N) -> seconds
ERRORS = {}  # (name, N) -> message
GRIDS = {}  # N -> grid spacing

for N in RESOLUTIONS:
    t0 = time.perf_counter()
    x, y, z, bx, by, bz = create_bfield(N)
    GRIDS[N] = x[1] - x[0]
    t_field = time.perf_counter() - t0
    print(f"\nN = {N}: field built in {t_field:.2f} s (h = {GRIDS[N]:.4f})")
    nrep = N_REPEAT_SMALL if N <= 128 else 1
    for name, spec in FINDERS.items():
        try:
            best = np.inf
            for _ in range(nrep):
                t0 = time.perf_counter()
                out = spec["run"](bx, by, bz, x, y, z)
                best = min(best, time.perf_counter() - t0)
            RESULTS[name, N], TIMES[name, N] = out, best
            print(f"  {name:10s} {len(out['pos']):3d} nulls  {best:8.4f} s")
        except Exception as e:
            ERRORS[name, N] = f"{type(e).__name__}: {e}"
            print(f"  {name:10s} FAILED: {ERRORS[name, N]}")
    del bx, by, bz

## 7. Comparison with the exact nulls

Detected nulls are assigned one-to-one to the exact ones by minimum total distance
(Hungarian algorithm).

* An exact null counts as **recovered** if its assigned null lies within
  `MATCH_TOL_CELLS × h` (default `0.5 × h`).
* Detected nulls that are not assigned within this same threshold are **spurious**.
* "type ok" compares the detected type with Table A.2.
* "charge ok" compares the detected charge with the charge implied by the exact type
  (A/As: +1, B/Bs: −1).

In [ ]:
def match_to_exact(out, h, match_tol_cells=MATCH_TOL_CELLS):
    found = out["pos"]
    m = pd.DataFrame(index=EXACT_NULLS.index)
    m["type"], m["err"], m["charge"], m["idx"] = "—", np.nan, np.nan, -1
    if len(found) == 0:
        return m, 0
    D = np.linalg.norm(exact_pos[:, None, :] - found[None, :, :], axis=-1)
    rows, cols = linear_sum_assignment(D)
    for r, c in zip(rows, cols, strict=True):
        if D[r, c] <= match_tol_cells * h:
            m.loc[r, ["type", "err", "charge", "idx"]] = (
                out["type"][c],
                D[r, c],
                out["charge"][c],
                c,
            )
    n_spurious = len(found) - int((m["idx"] >= 0).sum())
    return m, n_spurious


MATCHES, SPURIOUS = {}, {}
for key, out in RESULTS.items():
    MATCHES[key], SPURIOUS[key] = match_to_exact(out, GRIDS[key[1]])

exact_charge = EXACT_NULLS["exact_type"].map(TYPE_CHARGE).to_numpy(float)
rows = []
for (name, N), m in MATCHES.items():
    ok = m["idx"] >= 0
    rows.append(
        {
            "implementation": name,
            "N": N,
            "h": GRIDS[N],
            "found": len(RESULTS[name, N]["pos"]),
            "recovered (of 14)": int(ok.sum()),
            "type ok": int((m["type"] == EXACT_NULLS["exact_type"]).sum()),
            "charge ok": int((m["charge"].to_numpy() == exact_charge).sum())
            if ok.any()
            else 0,
            "spurious": SPURIOUS[name, N],
            "median err": m.loc[ok, "err"].median(),
            "max err": m.loc[ok, "err"].max(),
            "max err / h": m.loc[ok, "err"].max() / GRIDS[N],
            "time [s]": TIMES[name, N],
        }
    )
for (name, N), msg in ERRORS.items():
    rows.append(
        {"implementation": name, "N": N, "h": GRIDS.get(N, np.nan), "error": msg}
    )

SUMMARY = pd.DataFrame(rows).sort_values(["N", "implementation"]).reset_index(drop=True)
SUMMARY.round(4)

### Per-null detail

A type shown as `X≠Y` means the implementation found type X where Table A.2 lists Y.
`—` means the exact null was not recovered within `MATCH_TOL_CELLS × h`.
Chiti's types are
computed by the notebook from her own trilinear Jacobian (section 3).

In [ ]:
def per_null_table(N):
    t = EXACT_NULLS[["null_id", "exact_type"]].copy()
    for name in FINDERS:
        if (name, N) not in MATCHES:
            continue
        m = MATCHES[name, N]
        t[f"type {name}"] = [
            ft if ft in ("—", et) else f"{ft}≠{et}"
            for ft, et in zip(m["type"], EXACT_NULLS["exact_type"], strict=True)
        ]
        t[f"err {name}"] = m["err"]
    return t


for N in RESOLUTIONS:
    print(f"N = {N}  (h = {GRIDS.get(N, np.nan):.4f})")
    display(per_null_table(N).round(5))

## 8. Agreement between implementations

For each pair of implementations and each resolution, the detected nulls are matched
one-to-one, within `MATCH_TOL_CELLS × h` (half a grid spacing by default). The trilinear interpolant of the grid has exactly
defined nulls, so two correct trilinear finders should agree up to their convergence
tolerance. Larger differences point to a different interpolation, a missed root or a
looser tolerance.

In [ ]:
pair_rows = []
names = list(FINDERS)
for N in RESOLUTIONS:
    for i, a in enumerate(names):
        for b in names[i + 1 :]:
            if (a, N) not in RESULTS or (b, N) not in RESULTS:
                continue
            pa, pb = RESULTS[a, N]["pos"], RESULTS[b, N]["pos"]
            if len(pa) == 0 or len(pb) == 0:
                continue
            D = np.linalg.norm(pa[:, None] - pb[None], axis=-1)
            r, c = linear_sum_assignment(D)
            keep = D[r, c] <= MATCH_TOL_CELLS * GRIDS[N]
            r, c = r[keep], c[keep]
            ta = np.array(RESULTS[a, N]["type"])[r]
            tb = np.array(RESULTS[b, N]["type"])[c]
            dmax = D[r, c].max() if len(r) else np.nan
            pair_rows.append(
                {
                    "N": N,
                    "A": a,
                    "B": b,
                    "n A": len(pa),
                    "n B": len(pb),
                    "common": len(r),
                    "only A": len(pa) - len(r),
                    "only B": len(pb) - len(r),
                    "max |Δpos|": f"{dmax:.2e}",
                    "max |Δpos| / h": f"{dmax / GRIDS[N]:.2e}",
                    "type disagreements": int((ta != tb).sum()),
                }
            )
AGREEMENT = pd.DataFrame(pair_rows)
AGREEMENT

## 9. Convergence of the position error

For a smooth field the nulls of the trilinear interpolant converge as O(h²). The error
saturates at about 1e-4 because of the printed precision of Table A.2.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
# Decreasing marker sizes, so that coinciding results remain visible.
for k, name in enumerate(FINDERS):
    Ns = [N for N in RESOLUTIONS if (name, N) in MATCHES]
    if not Ns:
        continue
    h = np.array([GRIDS[N] for N in Ns])
    med = [MATCHES[name, N]["err"].median() for N in Ns]
    mx = [MATCHES[name, N]["err"].max() for N in Ns]
    ms = 10 - 3 * k
    (line,) = ax.loglog(h, med, "o-", ms=ms, label=f"{name} (median)")
    ax.loglog(
        h, mx, "s--", ms=ms, color=line.get_color(), alpha=0.7, label=f"{name} (max)"
    )
hh = np.array(sorted(GRIDS.values()))
anchor = [
    MATCHES[n, max(GRIDS, key=GRIDS.get)]["err"].median()
    for n in FINDERS
    if (n, max(GRIDS, key=GRIDS.get)) in MATCHES
]
if len(hh) and anchor:
    ax.loglog(hh, anchor[0] * (hh / hh.max()) ** 2, "k:", label="∝ h²")
ax.axhline(1e-4, color="0.6", lw=0.8)
ax.text(
    hh.max() if len(hh) else 0.3,
    1.15e-4,
    "Table A.2 precision",
    color="0.4",
    fontsize=8,
    ha="right",
    va="bottom",
)
ax.set_xlabel("grid spacing h")
ax.set_ylabel("distance to exact null")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)
plt.show()

## 10. 3D view of the detected nulls

The view uses the finest resolution. Exact nulls are drawn as black circles and the
detections of each implementation as coloured markers; no field lines are drawn.

In [ ]:
N_view = max(RESOLUTIONS)
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(projection="3d")
ax.scatter(
    *exact_pos.T,
    s=140,
    facecolors="none",
    edgecolors="k",
    linewidths=1.5,
    label="exact (Table A.2)",
)
for i, row in EXACT_NULLS.iterrows():
    ax.text(
        exact_pos[i, 0] + 0.12,
        exact_pos[i, 1],
        exact_pos[i, 2],
        f"{row.null_id} {row.exact_type}",
        fontsize=7,
    )
# Decreasing marker sizes, so that coinciding detections remain visible.
for k, (name, mk) in enumerate(zip(FINDERS, ["o", "^", "s", "D"], strict=False)):
    if (name, N_view) in RESULTS:
        p = RESULTS[name, N_view]["pos"]
        ax.scatter(
            *p.T, s=60 / (k + 1), marker=mk, alpha=0.8, label=f"{name} ({len(p)})"
        )
lim = (-np.pi / 2, 3 * np.pi / 2)
ax.set(
    xlim=lim,
    ylim=lim,
    zlim=lim,
    xlabel="x",
    ylabel="y",
    zlabel="z",
    title=f"Detected nulls, N = {N_view}",
)
ax.legend(fontsize=8, loc="upper left")
plt.show()